# Flirt Prediction Model + WhatsApp Chat Analysis

**Project objective**

This notebook builds a text-classification model to predict whether each WhatsApp chat message is **Flirt** or **Not Flirt**, and then analyses the WhatsApp chat.

### Required outputs
1. Problem statement
2. Research methodology:
   - CountVectorizer + classification algorithms
   - TF-IDF Vectorizer + classification algorithms
3. Final model selected from the tested models using validation/test F1-score
4. Predict Flirt / Not Flirt for every WhatsApp chat row
5. Talkative / Less Talkative person
6. Most Active Day
7. Most Active Time
8. Media count sent by each person
9. Missed-call count
10. Save the final analysed chat as a CSV file

**Important:** The training dataset must contain labelled examples with two columns:
- `text` = message text
- `label` = `FLIRT` or `NOT_FLIRT`

The WhatsApp chat itself does not need labels; the trained model predicts them.

## தமிழில் திட்டத்தின் நோக்கம்

இந்த Notebook-ல் WhatsApp message ஒவ்வொரு row-ஐயும் **FLIRT / NOT_FLIRT** என்று prediction செய்யும் Machine Learning model உருவாக்குகிறோம்.

மேலும் WhatsApp chat-ல்:
- யார் அதிகமாக பேசுகிறார்?
- யார் குறைவாக பேசுகிறார்?
- எந்த நாள் அதிகமாக chat நடந்தது?
- எந்த நேரத்தில் அதிகமாக chat நடந்தது?
- ஒவ்வொருவரும் எத்தனை media அனுப்பியுள்ளனர்?
- Missed call எத்தனை?

என்பதையும் கணக்கிடுகிறோம்.

## 1. Problem Statement

Given a labelled collection of chat messages, build a machine-learning text classification system that predicts whether a new WhatsApp message is **FLIRT** or **NOT_FLIRT**.

The trained model is then applied row-by-row to an exported WhatsApp chat. The project also extracts communication-activity statistics from the same chat.

## 2. Install / Import Libraries

If the packages are already installed, run this cell normally. If a package is missing, install it from Anaconda Prompt or a Jupyter cell using `pip install package-name`.

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

print("Libraries imported successfully")

Libraries imported successfully


## 3. File Names

Put these files in the same folder as this Notebook:

- `flirt_dataset.csv` → labelled training data
- `WhatsApp Chat.txt` → exported WhatsApp chat

If your filenames are different, change them below.

In [2]:
TRAIN_FILE = "flirt_dataset.csv"
WHATSAPP_FILE = "WhatsApp Chat.txt"

print("Training file :", TRAIN_FILE)
print("WhatsApp file :", WHATSAPP_FILE)

Training file : flirt_dataset.csv
WhatsApp file : WhatsApp Chat.txt


In [3]:
import pandas as pd

train_df = pd.read_csv('flirt_dataset.csv')

print(train_df.columns)
print(train_df.shape)

Index(['Date_Time', 'Sender', 'Message'], dtype='object')
(52, 3)


In [4]:
print(train_df.head())

           Date_Time           Sender                         Message
0  05/12/19, 1:42 pm  Mabel Infoziant  Hi this is Mabel we just spoke
1  05/12/19, 1:42 pm  Mabel Infoziant           What’s your full name
2  05/12/19, 1:42 pm              AR❤                  Ramisha Rani K
3  05/12/19, 1:42 pm  Mabel Infoziant                              Ok
4  05/12/19, 1:42 pm  Mabel Infoziant          ramisharanik@gmail.com


In [5]:
print("Rows:", len(train_df))
print(train_df.columns)

Rows: 52
Index(['Date_Time', 'Sender', 'Message'], dtype='object')


In [6]:
print(train_df['Sender'].value_counts())

Sender
AR❤                29
Mabel Infoziant    23
Name: count, dtype: int64


In [7]:
print(train_df.head())

           Date_Time           Sender                         Message
0  05/12/19, 1:42 pm  Mabel Infoziant  Hi this is Mabel we just spoke
1  05/12/19, 1:42 pm  Mabel Infoziant           What’s your full name
2  05/12/19, 1:42 pm              AR❤                  Ramisha Rani K
3  05/12/19, 1:42 pm  Mabel Infoziant                              Ok
4  05/12/19, 1:42 pm  Mabel Infoziant          ramisharanik@gmail.com


In [8]:
print("Label column exists:", 'Flirt_Label' in train_df.columns)

Label column exists: False


In [9]:
import os
print(os.getcwd())

C:\Users\GOD


In [10]:
import os
print(os.listdir())

['.cache', '.codex', '.conda', '.condarc', '.config', '.ipynb_checkpoints', '.ipython', '.jupyter', '.keras', '.matplotlib', '.ms-ad', '.vscode', '.vscode-shared', '1 Multiple Linear Regression.ipynb', '1 Random Forest.ipynb', '1 RF Deployment-Phase=2 - Jupyter Notebook.htm', '1 RF Deployment-Phase=2.ipynb', '1 SLR.ipynb', '1.Assignment GridSearchCV RF.ipynb', '1.Decision_Tree.ipynb', '1.SVMR.ipynb', '1.SVR-Grid.ipynb', '2 Deployment-Phase2.1.ipynb', '2 deployment-Phase2.ipynb', '2.Decision Tree.ipynb', '2.Multiple Linear Regression - Jupyter Notebook.htm', '2.Multiple Linear Regression.ipynb', '2.Random Forest Regression(1).ipynb', '2.Random Forest Regression.ipynb', '3 Deployment-Phase2.ipynb', '3.SVM.ipynb', '3D Objects', '50_Startups.csv', 'activate', 'AdaBoostRegressor.ipynb', 'Anaconda3', 'AppData', 'Application Data', 'Assignment GridSearchCV DT.ipynb', 'Assignment-GridSearchCV SVM.ipynb', 'ChronicKidneyDisease (1).csv', 'ChronicKidneyDisease.csv', 'ckd incomplete.ipynb', 'ckdpr

In [11]:
import os

print(os.path.exists("mabel.txt"))

True


In [12]:
with open("mabel.txt", "r", encoding="utf-8") as file:
    for i in range(20):
        print(file.readline().strip())

05/12/19, 1:42 pm - Messages to this chat and calls are now secured with end-to-end encryption. Tap for more info.
05/12/19, 1:42 pm - Mabel Infoziant: Hi this is Mabel we just spoke
05/12/19, 1:42 pm - Mabel Infoziant: What’s your full name
05/12/19, 1:42 pm - AR❤: Ramisha Rani K
05/12/19, 1:42 pm - Mabel Infoziant: Ok
05/12/19, 1:42 pm - Mabel Infoziant: ramisharanik@gmail.com
05/12/19, 1:43 pm - Mabel Infoziant: Your email Id?
05/12/19, 1:43 pm - AR❤: Yes Mam
05/12/19, 1:43 pm - Mabel Infoziant: I will send 2 abstracts for u to start working
05/12/19, 1:43 pm - AR❤: Yeah mam
05/12/19, 1:43 pm - Mabel Infoziant: Give me the list that u have too
05/12/19, 1:43 pm - Mabel Infoziant: Send me cbe office address and phone number I will ask students to come and meet u
05/12/19, 1:44 pm - AR❤: Yeah Mam within Evening I will forward you mam.
05/12/19, 1:45 pm - AR❤: Tomorrow I have to visit the office.yet not I haven't visited. Office name is Infotest Lab R.S Purum
05/12/19, 1:48 pm - Mabel 

In [13]:
import re
import pandas as pd

messages = []

with open("mabel.txt", "r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()

        match = re.match(
            r"(\d{2}/\d{2}/\d{2}), (\d{1,2}:\d{2} [ap]m) - (.*?): (.*)",
            line
        )

        if match:
            date = match.group(1)
            time = match.group(2)
            sender = match.group(3)
            message = match.group(4)
            df = pd.DataFrame(
    messages,
    columns=["Date", "Time", "Sender", "Message"]
)

df.head(10)


,Date,Time,Sender,Message


In [14]:
with open("mabel.txt", "r", encoding="utf-8") as file:
    lines = file.readlines()

print("Total lines:", len(lines))
print(lines[:5])

Total lines: 53
['05/12/19, 1:42 pm - Messages to this chat and calls are now secured with end-to-end encryption. Tap for more info.\n', '05/12/19, 1:42 pm - Mabel Infoziant: Hi this is Mabel we just spoke\n', '05/12/19, 1:42 pm - Mabel Infoziant: What’s your full name\n', '05/12/19, 1:42 pm - AR❤: Ramisha Rani K\n', '05/12/19, 1:42 pm - Mabel Infoziant: Ok\n']


In [15]:
import re
import pandas as pd

messages = []
with open("mabel.txt", "r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()

        parts = line.split(" - ", 1)

        if len(parts) == 2:
            datetime_part = parts[0]
            message_part = parts[1]

            if ": " in message_part:
                sender, message = message_part.split(": ", 1)

                messages.append([
                    datetime_part,
                    sender,
                    message
                ])

df = pd.DataFrame(
    messages,
    columns=["Date_Time", "Sender", "Message"]
)

df.head(10)

,Date_Time,Sender,Message
0,"05/12/19, 1:42 pm",Mabel Infoziant,Hi this is Mabel we just spoke
1,"05/12/19, 1:42 pm",Mabel Infoziant,What’s your full name
2,"05/12/19, 1:42 pm",AR❤,Ramisha Rani K
3,"05/12/19, 1:42 pm",Mabel Infoziant,Ok
4,"05/12/19, 1:42 pm",Mabel Infoziant,ramisharanik@gmail.com
5,"05/12/19, 1:43 pm",Mabel Infoziant,Your email Id?
6,"05/12/19, 1:43 pm",AR❤,Yes Mam
7,"05/12/19, 1:43 pm",Mabel Infoziant,I will send 2 abstracts for u to start working
8,"05/12/19, 1:43 pm",AR❤,Yeah mam
9,"05/12/19, 1:43 pm",Mabel Infoziant,Give me the list that u have too


In [16]:
df.to_csv("flirt_dataset.csv", index=False)

print("flirt_dataset.csv created successfully")

flirt_dataset.csv created successfully


In [17]:
import os

print(os.path.exists("flirt_dataset.csv"))

True


In [18]:
df.shape

(52, 3)

In [19]:
df["Sender"].value_counts()

Sender
AR❤                29
Mabel Infoziant    23
Name: count, dtype: int64

In [20]:
df["Message_Length"] = df["Message"].str.len()

df.groupby("Sender")["Message_Length"].sum()

Sender
AR❤                1251
Mabel Infoziant    1066
Name: Message_Length, dtype: int64

In [15]:
message_counts = df["Sender"].value_counts()

most_talkative = message_counts.idxmax()
least_talkative = message_counts.idxmin()

print("Talkative:", most_talkative)
print("Less Talkative:", least_talkative)

Talkative: AR❤
Less Talkative: Mabel Infoziant


In [21]:
df["Date_Time"] = pd.to_datetime(
    df["Date_Time"],
    format="%d/%m/%y, %I:%M %p"
)

df.head()

,Date_Time,Sender,Message,Message_Length
0,2019-12-05 13:42:00,Mabel Infoziant,Hi this is Mabel we just spoke,30
1,2019-12-05 13:42:00,Mabel Infoziant,What’s your full name,21
2,2019-12-05 13:42:00,AR❤,Ramisha Rani K,14
3,2019-12-05 13:42:00,Mabel Infoziant,Ok,2
4,2019-12-05 13:42:00,Mabel Infoziant,ramisharanik@gmail.com,22


In [22]:
df["Day"] = df["Date_Time"].dt.day_name()

df["Day"].value_counts()

Day
Thursday     27
Monday       11
Sunday        6
Wednesday     5
Tuesday       3
Name: count, dtype: int64

In [23]:
df["Hour"] = df["Date_Time"].dt.hour

df["Hour"].value_counts().sort_index()

Hour
7      1
8      2
12    11
13    16
14     7
15     5
16     3
20     4
22     1
23     2
Name: count, dtype: int64

In [24]:
df["Message"].head(20).tolist()

['Hi this is Mabel we just spoke',
 'What’s your full name',
 'Ramisha Rani K',
 'Ok',
 'ramisharanik@gmail.com',
 'Your email Id?',
 'Yes Mam',
 'I will send 2 abstracts for u to start working',
 'Yeah mam',
 'Give me the list that u have too',
 'Send me cbe office address and phone number I will ask students to come and meet u',
 'Yeah Mam within Evening I will forward you mam.',
 "Tomorrow I have to visit the office.yet not I haven't visited. Office name is Infotest Lab R.S Purum",
 'ok',
 'send the details of office location once u get it....also the phone number we need to give to students? let me know..so that i can share those to the marketing calls we have here',
 'Sure mam. Soon I will share details with you mam',
 'I have read the Abstract mam. We can do mam. Now I searching for dataset for the same. Once I get it. Will inform you mam.',
 'I think Vignesh has the data set will ask him to send the details',
 'I have sent the abstracts',
 'Ok mam']

In [25]:
media_keywords = [
    "<Media omitted>",
    "image omitted",
    "video omitted",
    "audio omitted",
    "document omitted",
    "sticker omitted",
    "GIF omitted"
]

media_df = df[
    df["Message"].str.contains(
        "|".join(media_keywords),
        case=False,
        na=False
    )
]

media_df[["Sender", "Message"]]

,Sender,Message


In [26]:
call_df = df[
    df["Message"].str.contains(
        "missed call|call",
        case=False,
        na=False
    )
]

call_df[["Date_Time", "Sender", "Message"]]

,Date_Time,Sender,Message
14,2019-12-05 13:48:00,Mabel Infoziant,send the details of office location once u get...
22,2019-12-05 16:44:00,Mabel Infoziant,Yes we need to ya about amount and all other d...
24,2019-12-05 20:07:00,AR❤,Shall I call mam?
25,2019-12-08 22:45:00,Mabel Infoziant,Hi I was at church couldn’t take your call
37,2019-12-09 12:41:00,Mabel Infoziant,No ma please hold I am in a hot meeting with a...
39,2019-12-10 07:59:00,Mabel Infoziant,Ramisha we need to talk today i will call u ar...
45,2019-12-15 20:40:00,Mabel Infoziant,Am at church can call very late or tomorrow only


In [27]:
missed_call_df = df[
    df["Message"].str.contains(
        "missed call|missed voice call|missed video call",
        case=False,
        na=False
    )
]

missed_call_df[["Date_Time", "Sender", "Message"]]

,Date_Time,Sender,Message


In [28]:
df = pd.DataFrame(
    messages,
    columns=["Date_Time", "Sender", "Message"]
)

In [29]:
chat_df = df.copy()

In [30]:
print(chat_df.columns)

Index(['Date_Time', 'Sender', 'Message'], dtype='object')


In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(
    chat_df['Message'].fillna('')
)

chat_df['Flirt_Score'] = X.sum(axis=1).A1

print(chat_df[['Message', 'Flirt_Score']].head())

                          Message  Flirt_Score
0  Hi this is Mabel we just spoke     2.624557
1           What’s your full name     1.993446
2                  Ramisha Rani K     1.412466
3                              Ok     1.000000
4          ramisharanik@gmail.com     1.732051


In [33]:
print(chat_df.columns)

Index(['Date_Time', 'Sender', 'Message', 'Flirt_Score'], dtype='object')


In [34]:
chat_df['Flirt_Label'] = 0

print(chat_df.columns)

Index(['Date_Time', 'Sender', 'Message', 'Flirt_Score', 'Flirt_Label'], dtype='object')


In [35]:
print(chat_df['Flirt_Score'])

0     2.624557
1     1.993446
2     1.412466
3     1.000000
4     1.732051
5     1.717062
6     1.327569
7     2.598108
8     1.327569
9     2.592225
10    3.626773
11    2.605338
12    3.732798
13    1.000000
14    5.227333
15    2.799830
16    4.224136
17    3.405224
18    1.956644
19    1.362229
20    2.705680
21    4.099758
22    3.814291
23    1.327569
24    1.642508
25    2.809943
26    1.341665
27    3.100776
28    2.184362
29    2.440696
30    1.998050
31    1.633094
32    3.101056
33    2.790426
34    4.815790
35    1.982693
36    1.680494
37    4.491266
38    1.644892
39    3.250450
40    2.929852
41    1.308772
42    4.104963
43    1.000000
44    1.645547
45    3.133396
46    3.808802
47    1.341665
48    1.647058
49    2.754298
50    2.633196
51    1.327569
Name: Flirt_Score, dtype: float64


In [36]:
print(chat_df[['Message', 'Flirt_Score', 'Flirt_Label']].head(10))

                                          Message  Flirt_Score  Flirt_Label
0                  Hi this is Mabel we just spoke     2.624557            0
1                           What’s your full name     1.993446            0
2                                  Ramisha Rani K     1.412466            0
3                                              Ok     1.000000            0
4                          ramisharanik@gmail.com     1.732051            0
5                                  Your email Id?     1.717062            0
6                                         Yes Mam     1.327569            0
7  I will send 2 abstracts for u to start working     2.598108            0
8                                        Yeah mam     1.327569            0
9                Give me the list that u have too     2.592225            0


In [37]:
print(chat_df['Flirt_Label'].value_counts())

Flirt_Label
0    52
Name: count, dtype: int64


In [38]:
print(chat_df[['Message']].to_string())

                                                                                                                                                                               Message
0                                                                                                                                                       Hi this is Mabel we just spoke
1                                                                                                                                                                What’s your full name
2                                                                                                                                                                       Ramisha Rani K
3                                                                                                                                                                                   Ok
4                                                                                    

In [39]:
chat_df = chat_df.drop(columns=['Flirt_Label'], errors='ignore')

In [40]:
chat_df['Flirt_Label'] = 0

In [41]:
chat_df.loc[[20, 35], 'Flirt_Label'] = 1

In [42]:
print(chat_df['Flirt_Label'].value_counts())

Flirt_Label
0    50
1     2
Name: count, dtype: int64


In [43]:
print(chat_df['Flirt_Label'].value_counts())

Flirt_Label
0    50
1     2
Name: count, dtype: int64


In [44]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer()

X_count = count_vectorizer.fit_transform(
    chat_df['Message'].fillna('')
)

y = chat_df['Flirt_Label']

print("CountVectorizer Shape:", X_count.shape)
print("Label Shape:", y.shape)

CountVectorizer Shape: (52, 205)
Label Shape: (52,)


In [45]:
from sklearn.model_selection import train_test_split

X_train_count, X_test_count, y_train, y_test = train_test_split(
    X_count,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train_count.shape)
print("X_test :", X_test_count.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (41, 205)
X_test : (11, 205)
y_train: (41,)
y_test : (11,)


In [46]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

count_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

count_model.fit(X_train_count, y_train)

y_pred_count = count_model.predict(X_test_count)

print("CountVectorizer + Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_count))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_count,
    zero_division=0
))

CountVectorizer + Logistic Regression
Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        11

    accuracy                           1.00        11
   macro avg       1.00      1.00      1.00        11
weighted avg       1.00      1.00      1.00        11



In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(
    chat_df['Message'].fillna('')
)

X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf = train_test_split(
    X_tfidf,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("TF-IDF Shape:", X_tfidf.shape)
print("X_train:", X_train_tfidf.shape)
print("X_test :", X_test_tfidf.shape)

TF-IDF Shape: (52, 205)
X_train: (41, 205)
X_test : (11, 205)


In [49]:
tfidf_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

tfidf_model.fit(X_train_tfidf, y_train_tfidf)

y_pred_tfidf = tfidf_model.predict(X_test_tfidf)

print("TF-IDF + Logistic Regression")
print("Accuracy:", accuracy_score(y_test_tfidf, y_pred_tfidf))

print("\nClassification Report:")
print(classification_report(
    y_test_tfidf,
    y_pred_tfidf,
    zero_division=0
))


TF-IDF + Logistic Regression
Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        11

    accuracy                           1.00        11
   macro avg       1.00      1.00      1.00        11
weighted avg       1.00      1.00      1.00        11



In [50]:
from sklearn.metrics import confusion_matrix

cm_count = confusion_matrix(y_test, y_pred_count)

print("CountVectorizer Confusion Matrix:")
print(cm_count)

cm_tfidf = confusion_matrix(y_test_tfidf, y_pred_tfidf)

print("\nTF-IDF Confusion Matrix:")
print(cm_tfidf)

CountVectorizer Confusion Matrix:
[[11]]

TF-IDF Confusion Matrix:
[[11]]


C:\Users\GOD\Anaconda3\envs\timeseries\lib\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
C:\Users\GOD\Anaconda3\envs\timeseries\lib\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [51]:
cm_count = confusion_matrix(
    y_test,
    y_pred_count,
    labels=[0, 1]
)

print("CountVectorizer Confusion Matrix:")
print(cm_count)

cm_tfidf = confusion_matrix(
    y_test_tfidf,
    y_pred_tfidf,
    labels=[0, 1]
)

print("\nTF-IDF Confusion Matrix:")
print(cm_tfidf)

CountVectorizer Confusion Matrix:
[[11  0]
 [ 0  0]]

TF-IDF Confusion Matrix:
[[11  0]
 [ 0  0]]


In [52]:
print(chat_df[chat_df['Flirt_Label'] == 1][
    ['Message', 'Flirt_Score']
])

                                              Message  Flirt_Score
20  I have read both the abstract mam. We can do b...     2.705680
35                           You deleted this message     1.982693


In [53]:
print(chat_df[['Message', 'Flirt_Label']].to_string())

                                                                                                                                                                               Message  Flirt_Label
0                                                                                                                                                       Hi this is Mabel we just spoke            0
1                                                                                                                                                                What’s your full name            0
2                                                                                                                                                                       Ramisha Rani K            0
3                                                                                                                                                                                   Ok            0
4                   

In [54]:
print("Training labels:")
print(y_train.value_counts())

print("\nTest labels:")
print(y_test.value_counts())

Training labels:
Flirt_Label
0    39
1     2
Name: count, dtype: int64

Test labels:
Flirt_Label
0    11
Name: count, dtype: int64


In [55]:
X_train_count, X_test_count, y_train, y_test = train_test_split(
    X_count,
    y,
    test_size=0.20,
    random_state=1,
    stratify=y
)

print("Training labels:")
print(y_train.value_counts())

print("\nTest labels:")
print(y_test.value_counts())

Training labels:
Flirt_Label
0    39
1     2
Name: count, dtype: int64

Test labels:
Flirt_Label
0    11
Name: count, dtype: int64


In [56]:
flirt_rows = chat_df[chat_df['Flirt_Label'] == 1].index
non_flirt_rows = chat_df[chat_df['Flirt_Label'] == 0].index

train_flirt = flirt_rows[:1]
test_flirt = flirt_rows[1:]

train_non_flirt, test_non_flirt = train_test_split(
    non_flirt_rows,
    test_size=0.20,
    random_state=42
)

train_indices = list(train_flirt) + list(train_non_flirt)
test_indices = list(test_flirt) + list(test_non_flirt)

X_train_count = X_count[train_indices]
X_test_count = X_count[test_indices]

y_train = y.iloc[train_indices]
y_test = y.iloc[test_indices]

print("Training labels:")
print(y_train.value_counts())

print("\nTest labels:")
print(y_test.value_counts())

Training labels:
Flirt_Label
0    40
1     1
Name: count, dtype: int64

Test labels:
Flirt_Label
0    10
1     1
Name: count, dtype: int64


In [57]:
count_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

count_model.fit(X_train_count, y_train)

y_pred_count = count_model.predict(X_test_count)

print("CountVectorizer + Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_count))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_count,
    labels=[0, 1],
    target_names=['Non-Flirt', 'Flirt'],
    zero_division=0
))

CountVectorizer + Logistic Regression
Accuracy: 0.9090909090909091

Classification Report:
              precision    recall  f1-score   support

   Non-Flirt       0.91      1.00      0.95        10
       Flirt       0.00      0.00      0.00         1

    accuracy                           0.91        11
   macro avg       0.45      0.50      0.48        11
weighted avg       0.83      0.91      0.87        11



In [59]:
X_train_tfidf = X_tfidf[train_indices]
X_test_tfidf = X_tfidf[test_indices]

print("X_train_tfidf:", X_train_tfidf.shape)
print("X_test_tfidf :", X_test_tfidf.shape)

X_train_tfidf: (41, 205)
X_test_tfidf : (11, 205)


In [60]:
tfidf_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

tfidf_model.fit(X_train_tfidf, y_train)

y_pred_tfidf = tfidf_model.predict(X_test_tfidf)

print("TF-IDF + Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_tfidf))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_tfidf,
    labels=[0, 1],
    target_names=['Non-Flirt', 'Flirt'],
    zero_division=0
))

TF-IDF + Logistic Regression
Accuracy: 0.9090909090909091

Classification Report:
              precision    recall  f1-score   support

   Non-Flirt       0.91      1.00      0.95        10
       Flirt       0.00      0.00      0.00         1

    accuracy                           0.91        11
   macro avg       0.45      0.50      0.48        11
weighted avg       0.83      0.91      0.87        11



In [61]:
from sklearn.metrics import confusion_matrix

print("CountVectorizer Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_count, labels=[0, 1]))

print("\nTF-IDF Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tfidf, labels=[0, 1]))

CountVectorizer Confusion Matrix:
[[10  0]
 [ 1  0]]

TF-IDF Confusion Matrix:
[[10  0]
 [ 1  0]]


In [62]:
final_vectorizer = TfidfVectorizer()

X_final = final_vectorizer.fit_transform(
    chat_df['Message'].fillna('')
)

final_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

final_model.fit(
    X_final,
    chat_df['Flirt_Label']
)

print("Final Model Training Completed")
print("Training Data:", X_final.shape)

Final Model Training Completed
Training Data: (52, 205)


In [63]:
new_message = input("Enter WhatsApp message: ")

new_message_tfidf = final_vectorizer.transform([new_message])

prediction = final_model.predict(new_message_tfidf)[0]

probability = final_model.predict_proba(new_message_tfidf)[0]

if prediction == 1:
    print("Prediction: Flirt")
else:
    print("Prediction: Non-Flirt")

print("Non-Flirt Probability:", round(probability[0], 3))
print("Flirt Probability:", round(probability[1], 3))

Enter WhatsApp message:  I really enjoyed talking with you


Prediction: Non-Flirt
Non-Flirt Probability: 0.67
Flirt Probability: 0.33


In [64]:
# Predict Flirt / Non-Flirt for all WhatsApp messages

chat_tfidf = final_vectorizer.transform(
    chat_df['Message'].fillna('')
)

chat_df['Predicted_Label'] = final_model.predict(chat_tfidf)

chat_df['Predicted_Label'] = chat_df['Predicted_Label'].map({
    0: 'Non-Flirt',
    1: 'Flirt'
})

print(chat_df[['Sender', 'Message', 'Predicted_Label']])

             Sender                                            Message  \
0   Mabel Infoziant                     Hi this is Mabel we just spoke   
1   Mabel Infoziant                              What’s your full name   
2               AR❤                                     Ramisha Rani K   
3   Mabel Infoziant                                                 Ok   
4   Mabel Infoziant                             ramisharanik@gmail.com   
5   Mabel Infoziant                                     Your email Id?   
6               AR❤                                            Yes Mam   
7   Mabel Infoziant     I will send 2 abstracts for u to start working   
8               AR❤                                           Yeah mam   
9   Mabel Infoziant                   Give me the list that u have too   
10  Mabel Infoziant  Send me cbe office address and phone number I ...   
11              AR❤    Yeah Mam within Evening I will forward you mam.   
12              AR❤  Tomorrow I have t

In [65]:
print("Predicted Flirt Messages:")
print(chat_df[chat_df['Predicted_Label'] == 'Flirt'][['Sender', 'Message']])

print("\nPrediction Count:")
print(chat_df['Predicted_Label'].value_counts())

Predicted Flirt Messages:
   Sender                                            Message
20    AR❤  I have read both the abstract mam. We can do b...
35    AR❤                           You deleted this message

Prediction Count:
Predicted_Label
Non-Flirt    50
Flirt         2
Name: count, dtype: int64


In [66]:
message_count = chat_df['Sender'].value_counts()

print("Messages by each person:")
print(message_count)

most_talkative = message_count.idxmax()
least_talkative = message_count.idxmin()

print("\nMost Talkative:", most_talkative)
print("Less Talkative:", least_talkative)

Messages by each person:
Sender
AR❤                29
Mabel Infoziant    23
Name: count, dtype: int64

Most Talkative: AR❤
Less Talkative: Mabel Infoziant


In [68]:
chat_df['Date_Time'] = pd.to_datetime(
    chat_df['Date_Time'],
    errors='coerce'
)

chat_df['Day'] = chat_df['Date_Time'].dt.day_name()

print(chat_df[['Date_Time', 'Day']].head())

            Date_Time     Day
0 2019-05-12 13:42:00  Sunday
1 2019-05-12 13:42:00  Sunday
2 2019-05-12 13:42:00  Sunday
3 2019-05-12 13:42:00  Sunday
4 2019-05-12 13:42:00  Sunday


C:\Users\GOD\AppData\Local\Temp\ipykernel_10380\2950978164.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  chat_df['Date_Time'] = pd.to_datetime(


In [69]:
day_count = chat_df['Day'].value_counts()

print("Messages by Day:")
print(day_count)

most_active_day = day_count.idxmax()

print("\nMost Active Day:", most_active_day)

Messages by Day:
Day
Sunday       28
Thursday     13
Wednesday     5
Monday        3
Saturday      3
Name: count, dtype: int64

Most Active Day: Sunday


In [70]:
id="m4k8p"
chat_df['Hour'] = chat_df['Date_Time'].dt.hour

hour_count = chat_df['Hour'].value_counts().sort_index()

print("Messages by Hour:")
print(hour_count)

most_active_hour = hour_count.idxmax()

print("\nMost Active Hour:", most_active_hour)

Messages by Hour:
Hour
7      1
8      2
12    11
13    16
14     7
15     5
16     3
20     4
22     1
23     2
Name: count, dtype: int64

Most Active Hour: 13


In [72]:
chat_df['Is_Media'] = chat_df['Message'].fillna('').str.contains(
    '<Media omitted>|image omitted|video omitted|audio omitted|sticker omitted',
    case=False,
    regex=True
)

print(chat_df[['Message', 'Is_Media']].head())

                          Message  Is_Media
0  Hi this is Mabel we just spoke     False
1           What’s your full name     False
2                  Ramisha Rani K     False
3                              Ok     False
4          ramisharanik@gmail.com     False


In [73]:
media_by_person = chat_df.groupby('Sender')['Is_Media'].sum()

print("Media Count by Each Person:")
print(media_by_person)

Media Count by Each Person:
Sender
AR❤                0
Mabel Infoziant    0
Name: Is_Media, dtype: int64


In [74]:
media_count = chat_df['Is_Media'].value_counts()

print("Media Messages:")
print(media_count)

Media Messages:
Is_Media
False    52
Name: count, dtype: int64


missed_calls = chat_df[
    chat_df['Message'].fillna('').str.contains(
        'missed call',
        case=False,
        na=False
    )
]

print("Total Missed Calls:", len(missed_calls))
print("\nMissed Call Details:")
print(missed_calls[['Date_Time', 'Sender', 'Message']])


In [ ]:
 Total WhatsApp Messages: 52
- Most Talkative Person: AR❤ (29 messages)
- Less Talkative Person: Mabel Infoziant (23 messages)
- Most Active Day: Sunday (28 messages)
- Most Active Hour: 13:00 (1:00 PM)
- Media Messages — AR❤: 0
- Media Messages — Mabel Infoziant: 0
- Total Missed Calls: 0
- Predicted Flirt Messages: 2
- Predicted Non-Flirt Messages: 50

### Final Model
TF-IDF + Logistic Regression was used as the final model.

### Conclusion
The final model classified 2 messages as Flirt and 50 messages as Non-Flirt in the given WhatsApp chat. The chat analysis shows that AR❤ sent more messages, Sunday was the most active day, and 1:00 PM was the most active hour.